# AG02 - LangChain Basics

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/nexageapps/AI/blob/main/Agents/AG02%20-%20LangChain%20Basics.ipynb)

**What you'll learn:**
- Installation and environment setup
- LLM wrappers and chat models
- Prompt templates and engineering
- Output parsers (String, JSON, Pydantic)
- Building chains with LCEL (LangChain Expression Language)
- Your first complete LangChain application

**Duration:** 1.5 hours

---

## 1. Installation and Setup

### Install Dependencies

In [ ]:
# Install core packages
!pip install -qU \
    langchain \
    langchain-openai \
    langchain-community \
    python-dotenv \
    openai

print("✅ Installation complete!")

### Set Up Environment Variables

**Option 1: Google Colab (using Secrets)**

In [ ]:
import os
from getpass import getpass

# For Colab: Use Secrets (left sidebar → key icon)
# Or enter manually (not recommended for production)

try:
    from google.colab import userdata
    os.environ["OPENAI_API_KEY"] = userdata.get('OPENAI_API_KEY')
    print("✅ Loaded API key from Colab Secrets")
except ImportError:
    # Local environment
    if not os.getenv("OPENAI_API_KEY"):
        os.environ["OPENAI_API_KEY"] = getpass("Enter your OpenAI API key: ")
    print("✅ API key set")

**Option 2: Local Development (using .env file)**

Create a `.env` file in your project root:
```
OPENAI_API_KEY=sk-...
```

Then load it:

In [ ]:
# For local development
from dotenv import load_dotenv
load_dotenv()  # Loads variables from .env file

# Verify
assert os.getenv("OPENAI_API_KEY"), "❌ API key not found!"
print("✅ Environment configured")

## 2. LLM Wrappers and Chat Models

### 2.1 Simple LLM Call

In [ ]:
from langchain_openai import ChatOpenAI

# Initialize the LLM
llm = ChatOpenAI(
    model="gpt-3.5-turbo",
    temperature=0.7,  # 0 = deterministic, 1 = creative
    max_tokens=None,  # None = no limit
)

# Simple invocation
response = llm.invoke("What is LangChain?")
print(response.content)

### 2.2 Chat Messages

LangChain uses message objects for chat:

In [ ]:
from langchain_core.messages import HumanMessage, SystemMessage, AIMessage

messages = [
    SystemMessage(content="You are a helpful AI assistant that explains concepts clearly."),
    HumanMessage(content="What is an AI agent in one sentence?")
]

response = llm.invoke(messages)
print(response.content)

### 2.3 Streaming Responses

For better UX, stream token-by-token:

In [ ]:
print("Streaming response:")
for chunk in llm.stream("Explain transformers in 50 words"):
    print(chunk.content, end="", flush=True)
print("\n\n✅ Streaming complete")

## 3. Prompt Templates

### 3.1 Basic String Template

In [ ]:
from langchain_core.prompts import PromptTemplate

# Create a template
template = PromptTemplate.from_template(
    "Tell me a {adjective} fact about {topic}"
)

# Format with variables
prompt = template.format(adjective="surprising", topic="Python")
print("Formatted prompt:")
print(prompt)
print("\nLLM response:")
print(llm.invoke(prompt).content)

### 3.2 Chat Prompt Template

Better for chat models:

In [ ]:
from langchain_core.prompts import ChatPromptTemplate

# Define a chat template
chat_template = ChatPromptTemplate.from_messages([
    ("system", "You are an expert {role}. Be concise and accurate."),
    ("human", "{question}")
])

# Format and invoke
messages = chat_template.format_messages(
    role="Python developer",
    question="What are decorators?"
)

print("Chat messages:")
for msg in messages:
    print(f"{msg.__class__.__name__}: {msg.content}")

response = llm.invoke(messages)
print(f"\nAIMessage: {response.content}")

### 3.3 Few-Shot Prompting

In [ ]:
from langchain_core.prompts import FewShotChatMessagePromptTemplate

# Examples
examples = [
    {"input": "2+2", "output": "4"},
    {"input": "3*3", "output": "9"},
]

# Example template
example_prompt = ChatPromptTemplate.from_messages([
    ("human", "{input}"),
    ("ai", "{output}")
])

# Few-shot template
few_shot_prompt = FewShotChatMessagePromptTemplate(
    example_prompt=example_prompt,
    examples=examples,
)

# Final template
final_prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a calculator. Give only the number."),
    few_shot_prompt,
    ("human", "{input}"),
])

# Test
messages = final_prompt.format_messages(input="5+7")
print(llm.invoke(messages).content)

## 4. Output Parsers

### 4.1 String Output Parser (Default)

In [ ]:
from langchain_core.output_parsers import StrOutputParser

parser = StrOutputParser()

# Chains: template → llm → parser
chain = chat_template | llm | parser

result = chain.invoke({
    "role": "data scientist",
    "question": "What is a p-value?"
})

print(type(result))  # str
print(result)

### 4.2 JSON Output Parser

In [ ]:
from langchain_core.output_parsers import JsonOutputParser

json_parser = JsonOutputParser()

json_template = ChatPromptTemplate.from_messages([
    ("system", "You are a data extractor. Return only valid JSON."),
    ("human", "Extract the name and age from: {text}. Format: {{\"name\": \"...\", \"age\": ...}}")
])

json_chain = json_template | llm | json_parser

result = json_chain.invoke({
    "text": "John Smith is 30 years old"
})

print(type(result))  # dict
print(result)
print(f"Name: {result['name']}, Age: {result['age']}")

### 4.3 Pydantic Output Parser (Type-Safe)

In [ ]:
from langchain_core.output_parsers import PydanticOutputParser
from pydantic import BaseModel, Field
from typing import List

# Define schema
class Person(BaseModel):
    name: str = Field(description="Person's full name")
    age: int = Field(description="Person's age in years")
    occupation: str = Field(description="Person's job")

# Create parser
pydantic_parser = PydanticOutputParser(pydantic_object=Person)

# Template with format instructions
pydantic_template = ChatPromptTemplate.from_messages([
    ("system", "Extract structured information.\n{format_instructions}"),
    ("human", "Extract: {text}")
])

# Chain
pydantic_chain = pydantic_template | llm | pydantic_parser

result = pydantic_chain.invoke({
    "text": "Alice Johnson, 28, works as a software engineer",
    "format_instructions": pydantic_parser.get_format_instructions()
})

print(type(result))  # Person
print(result)
print(f"\nAccess properties: {result.name} is {result.age} years old")

## 5. LCEL (LangChain Expression Language)

### 5.1 Basic Chain with | Operator

In [ ]:
# Simple chain: prompt | llm | parser
simple_chain = (
    PromptTemplate.from_template("Tell me a joke about {topic}")
    | llm
    | StrOutputParser()
)

result = simple_chain.invoke({"topic": "programming"})
print(result)

### 5.2 Sequential Chains

In [ ]:
# Chain 1: Generate a topic
topic_chain = (
    PromptTemplate.from_template("Suggest a {genre} movie topic in 3 words")
    | llm
    | StrOutputParser()
)

# Chain 2: Write a story about the topic
story_chain = (
    PromptTemplate.from_template("Write a 2-sentence story about: {topic}")
    | llm
    | StrOutputParser()
)

# Combined: topic_chain → story_chain
from langchain_core.runnables import RunnablePassthrough

full_chain = (
    {"topic": topic_chain}
    | story_chain
)

result = full_chain.invoke({"genre": "sci-fi"})
print(result)

### 5.3 Parallel Chains

In [ ]:
from langchain_core.runnables import RunnableParallel

# Create multiple chains
chain1 = (
    PromptTemplate.from_template("What are pros of {topic}?")
    | llm
    | StrOutputParser()
)

chain2 = (
    PromptTemplate.from_template("What are cons of {topic}?")
    | llm
    | StrOutputParser()
)

# Run in parallel
parallel_chain = RunnableParallel(pros=chain1, cons=chain2)

result = parallel_chain.invoke({"topic": "remote work"})
print("PROS:")
print(result["pros"])
print("\nCONS:")
print(result["cons"])

### 5.4 Fallback Chains

In [ ]:
from langchain_core.runnables import RunnableLambda

# Primary chain (might fail)
primary = llm

# Fallback chain
fallback = RunnableLambda(lambda x: "Sorry, I'm temporarily unavailable.")

# Chain with fallback
chain_with_fallback = primary.with_fallbacks([fallback])

# This will use primary if it works, fallback if it fails
result = chain_with_fallback.invoke("Hello")
print(result if isinstance(result, str) else result.content)

## 6. Complete Application: Sentiment Analyzer

Let's build a complete app that analyzes sentiment and extracts entities:

In [ ]:
from pydantic import BaseModel, Field
from typing import List, Literal

# Define output schema
class SentimentAnalysis(BaseModel):
    sentiment: Literal["positive", "negative", "neutral"] = Field(
        description="Overall sentiment of the text"
    )
    confidence: float = Field(
        description="Confidence score between 0 and 1"
    )
    entities: List[str] = Field(
        description="List of named entities (people, places, organizations)"
    )
    keywords: List[str] = Field(
        description="Key topics mentioned"
    )
    summary: str = Field(
        description="One-sentence summary"
    )

# Create parser
sentiment_parser = PydanticOutputParser(pydantic_object=SentimentAnalysis)

# Create template
sentiment_template = ChatPromptTemplate.from_messages([
    ("system", "You are a sentiment analysis expert.\n{format_instructions}"),
    ("human", "Analyze this text:\n\n{text}")
])

# Create chain
sentiment_chain = sentiment_template | llm | sentiment_parser

# Test with example text
test_text = """
Apple announced a fantastic new iPhone yesterday at their Cupertino headquarters.
The event was attended by tech journalists from around the world. CEO Tim Cook
presented impressive features including better battery life and improved cameras.
However, some analysts worry about the high price point of $1,200.
"""

result = sentiment_chain.invoke({
    "text": test_text,
    "format_instructions": sentiment_parser.get_format_instructions()
})

print("=== Sentiment Analysis Results ===")
print(f"Sentiment: {result.sentiment.upper()}")
print(f"Confidence: {result.confidence:.2%}")
print(f"Entities: {', '.join(result.entities)}")
print(f"Keywords: {', '.join(result.keywords)}")
print(f"Summary: {result.summary}")

## 7. Debugging and Monitoring

### 7.1 Verbose Mode

In [ ]:
# Enable verbose output
from langchain.globals import set_verbose, set_debug

set_verbose(True)
# set_debug(True)  # Even more detailed

simple_chain.invoke({"topic": "AI"})

set_verbose(False)  # Disable

### 7.2 Token Counting

In [ ]:
from langchain.callbacks import get_openai_callback

with get_openai_callback() as cb:
    result = sentiment_chain.invoke({
        "text": test_text,
        "format_instructions": sentiment_parser.get_format_instructions()
    })
    
    print("\n=== Token Usage ===")
    print(f"Prompt tokens: {cb.prompt_tokens}")
    print(f"Completion tokens: {cb.completion_tokens}")
    print(f"Total tokens: {cb.total_tokens}")
    print(f"Total cost: ${cb.total_cost:.4f}")

## 8. Different LLM Providers

LangChain supports many providers:

In [ ]:
# OpenAI (we've been using)
from langchain_openai import ChatOpenAI
openai_llm = ChatOpenAI(model="gpt-3.5-turbo")

# Anthropic Claude (requires: pip install langchain-anthropic)
# from langchain_anthropic import ChatAnthropic
# claude_llm = ChatAnthropic(model="claude-3-sonnet-20240229")

# Google (requires: pip install langchain-google-genai)
# from langchain_google_genai import ChatGoogleGenerativeAI
# google_llm = ChatGoogleGenerativeAI(model="gemini-pro")

# Local models via Ollama (free!)
# from langchain_community.llms import Ollama
# ollama_llm = Ollama(model="llama2")

print("✅ You can swap LLMs easily in LangChain!")

## 9. Practice Exercises

### Exercise 1: Translation Chain

Build a chain that:
1. Translates text to French
2. Then translates it back to English
3. Compares the result

**Solution:**

In [ ]:
# Your code here
# Hint: Create two sequential chains

# Example solution (uncomment to see)
"""
translate_to_french = (
    PromptTemplate.from_template("Translate to French: {text}")
    | llm | StrOutputParser()
)

translate_to_english = (
    PromptTemplate.from_template("Translate to English: {french}")
    | llm | StrOutputParser()
)

full_chain = (
    {"french": translate_to_french}
    | translate_to_english
)

result = full_chain.invoke({"text": "Hello, how are you?"})
print(result)
"""

### Exercise 2: Structured Extraction

Create a Pydantic model for a movie and extract information from text:

In [ ]:
# Your code here
# Define: class Movie(BaseModel) with title, year, genre, rating
# Create chain to extract from text
# Test with: "Inception (2010) is a sci-fi thriller rated 8.8/10"

# Example solution
"""
class Movie(BaseModel):
    title: str
    year: int
    genre: str
    rating: float

movie_parser = PydanticOutputParser(pydantic_object=Movie)
movie_template = ChatPromptTemplate.from_messages([
    ("system", "Extract movie info.\n{format_instructions}"),
    ("human", "{text}")
])

movie_chain = movie_template | llm | movie_parser

result = movie_chain.invoke({
    "text": "Inception (2010) is a sci-fi thriller rated 8.8/10",
    "format_instructions": movie_parser.get_format_instructions()
})
print(result)
"""

## 10. Summary

### Key Takeaways

1. **LangChain simplifies LLM application development**
2. **LCEL (`|` operator) makes chaining intuitive**
3. **Prompt templates** enable reusable, parameterized prompts
4. **Output parsers** structure LLM responses (String, JSON, Pydantic)
5. **Chains can be sequential, parallel, or have fallbacks**
6. **Always monitor tokens and costs**

### What's Next?

In **AG03 - Memory Systems**, you'll learn:
- Conversation buffer memory
- Summary memory for long conversations
- Vector store memory for semantic search
- Persistent memory strategies

---

### Quick Quiz

1. **What does LCEL stand for?**
   <details>
   <summary>Answer</summary>
   LangChain Expression Language - the syntax for chaining components with the | operator
   </details>

2. **When should you use Pydantic output parser instead of JSON parser?**
   <details>
   <summary>Answer</summary>
   When you need type safety, validation, and IDE autocomplete for structured outputs
   </details>

3. **How do you run two chains in parallel?**
   <details>
   <summary>Answer</summary>
   Use RunnableParallel: RunnableParallel(key1=chain1, key2=chain2)
   </details>

---

**Continue to AG03 to add memory! →**